In [3]:
# ==========================================================
# OpenAI Meeting Transcription with Speaker Diarization
# Supports meetings with multiple speakers
# ==========================================================

# Install:
# pip install -U openai python-dotenv

import json
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from openai import OpenAI


# ----------------------------------------------------------
# 1. Load API key
# ----------------------------------------------------------

load_dotenv()

api_key = os.getenv("OPENAI_API_KEY_L")

if not api_key:
    raise ValueError(
        "OPENAI_API_KEY is missing. Add it to your .env file."
    )

client = OpenAI(api_key=api_key)


# ----------------------------------------------------------
# 2. Configuration
# ----------------------------------------------------------

AUDIO_FILE = Path("ES2002a.Mix-Headset.wav")
OUTPUT_JSON = Path("diarized_transcript.json")
OUTPUT_TEXT = Path("diarized_transcript.txt")

# Supplying the language can improve accuracy and latency.
LANGUAGE = "en"


# ----------------------------------------------------------
# 3. Validate input file
# ----------------------------------------------------------

SUPPORTED_FORMATS = {
    ".flac",
    ".mp3",
    ".mp4",
    ".mpeg",
    ".mpga",
    ".m4a",
    ".ogg",
    ".wav",
    ".webm",
}

if not AUDIO_FILE.exists():
    raise FileNotFoundError(
        f"Audio file not found: {AUDIO_FILE.resolve()}"
    )

if AUDIO_FILE.suffix.lower() not in SUPPORTED_FORMATS:
    raise ValueError(
        f"Unsupported audio format: {AUDIO_FILE.suffix}. "
        f"Supported formats: {sorted(SUPPORTED_FORMATS)}"
    )


# ----------------------------------------------------------
# 4. Helper function
# ----------------------------------------------------------

def get_value(obj: Any, field: str, default: Any = None) -> Any:
    """
    Read a value from either an SDK object or a dictionary.
    """
    if isinstance(obj, dict):
        return obj.get(field, default)

    return getattr(obj, field, default)


def format_timestamp(seconds: float) -> str:
    """
    Convert seconds into HH:MM:SS.mmm format.
    """
    milliseconds = int((seconds % 1) * 1000)
    total_seconds = int(seconds)

    hours = total_seconds // 3600
    minutes = (total_seconds % 3600) // 60
    secs = total_seconds % 60

    return f"{hours:02d}:{minutes:02d}:{secs:02d}.{milliseconds:03d}"


# ----------------------------------------------------------
# 5. Send meeting audio for diarized transcription
# ----------------------------------------------------------

try:
    with AUDIO_FILE.open("rb") as audio_file:
        transcription = client.audio.transcriptions.create(
            model="gpt-4o-transcribe-diarize",
            file=audio_file,

            # Required for diarized speaker annotations
            response_format="diarized_json",

            # Required for diarization inputs longer than 30 seconds.
            # OpenAI applies loudness normalization and server-side VAD.
            chunking_strategy="auto",

            # Optional but useful when language is known
            language=LANGUAGE,
        )

except Exception as error:
    raise RuntimeError(
        f"OpenAI transcription failed: {error}"
    ) from error


# ----------------------------------------------------------
# 6. Extract diarized segments
# ----------------------------------------------------------

segments = get_value(transcription, "segments", [])
full_text = get_value(transcription, "text", "")
duration = get_value(transcription, "duration", 0.0)

if not segments:
    raise RuntimeError(
        "The API returned no diarized transcript segments."
    )


# ----------------------------------------------------------
# 7. Prepare structured output
# ----------------------------------------------------------

formatted_segments = []
detected_speakers = set()

for segment in segments:
    speaker = str(get_value(segment, "speaker", "UNKNOWN"))
    text = str(get_value(segment, "text", "")).strip()
    start = float(get_value(segment, "start", 0.0))
    end = float(get_value(segment, "end", 0.0))
    segment_id = str(get_value(segment, "id", ""))

    detected_speakers.add(speaker)

    formatted_segments.append(
        {
            "segment_id": segment_id,
            "speaker": speaker,
            "start_seconds": start,
            "end_seconds": end,
            "start_timestamp": format_timestamp(start),
            "end_timestamp": format_timestamp(end),
            "text": text,
        }
    )


result = {
    "audio_file": AUDIO_FILE.name,
    "model": "gpt-4o-transcribe-diarize",
    "duration_seconds": duration,
    "detected_speaker_count": len(detected_speakers),
    "detected_speakers": sorted(detected_speakers),
    "full_transcript": full_text,
    "segments": formatted_segments,
}


# ----------------------------------------------------------
# 8. Print speaker-attributed transcript
# ----------------------------------------------------------

print("=" * 80)
print("DIARIZED MEETING TRANSCRIPT")
print("=" * 80)

current_speaker = None

for segment in formatted_segments:
    speaker = segment["speaker"]
    start_time = segment["start_timestamp"]
    end_time = segment["end_timestamp"]
    text = segment["text"]

    if speaker != current_speaker:
        print(f"\n[{start_time} - {end_time}] Speaker {speaker}:")
        current_speaker = speaker
    else:
        print(f"[{start_time} - {end_time}]")

    print(text)

print("\n" + "=" * 80)
print(f"Detected speakers: {len(detected_speakers)}")
print(f"Speaker labels: {', '.join(sorted(detected_speakers))}")
print("=" * 80)


# ----------------------------------------------------------
# 9. Save complete JSON output
# ----------------------------------------------------------

with OUTPUT_JSON.open("w", encoding="utf-8") as json_file:
    json.dump(
        result,
        json_file,
        ensure_ascii=False,
        indent=2,
    )


# ----------------------------------------------------------
# 10. Save readable text transcript
# ----------------------------------------------------------

with OUTPUT_TEXT.open("w", encoding="utf-8") as text_file:
    text_file.write("DIARIZED MEETING TRANSCRIPT\n")
    text_file.write("=" * 80 + "\n\n")

    for segment in formatted_segments:
        text_file.write(
            f"[{segment['start_timestamp']} - "
            f"{segment['end_timestamp']}] "
            f"Speaker {segment['speaker']}:\n"
        )
        text_file.write(segment["text"] + "\n\n")


print(f"\nJSON saved to: {OUTPUT_JSON.resolve()}")
print(f"Text saved to: {OUTPUT_TEXT.resolve()}")

DIARIZED MEETING TRANSCRIPT

[00:00:04.596 - 00:00:07.346] Speaker A:
Thank gosh, you've already pres produced a PowerPoint presentation.

[00:00:07.346 - 00:00:09.396] Speaker B:
Uh I think it's already on actually.

[00:00:10.545 - 00:00:10.646] Speaker A:
Hmm.

[00:00:12.146 - 00:00:12.596] Speaker B:
Uh
[00:00:14.896 - 00:00:16.196]
God how did I make this thing work?

[00:00:19.295 - 00:00:19.946] Speaker C:
Should maybe the input.

[00:00:20.746 - 00:00:20.846] Speaker B:
Uh

[00:00:20.846 - 00:00:20.996] Speaker A:
Hmm.

[00:00:20.996 - 00:00:21.146] Speaker B:

[00:00:33.454 - 00:00:34.954]
I put it in the back, but

[00:00:38.103 - 00:00:38.954] Speaker D:
Yep, it's gone.

[00:00:39.804 - 00:00:41.904] Speaker B:
Okay, right. Kind of our

[00:00:42.204 - 00:00:42.754] Speaker E:
It's okay.
[00:00:43.004 - 00:00:43.054]
That's

[00:00:44.754 - 00:00:46.004] Speaker D:
It's doing something.
[00:00:48.103 - 00:00:48.254]
Yes.

[00:00:48.254 - 00:00:48.353] Speaker B:
Right.

[00:

#### straemimng

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=api_key)

with open("ES2002a.Mix-Headset.wav", "rb") as audio_file:

    stream = client.audio.transcriptions.create(
        file=audio_file,
        model="gpt-4o-transcribe-diarize",
        response_format="diarized_json",
        chunking_strategy="auto",
        language="en",
        temperature=0,
        stream=True,
    )

    for event in stream:

        if event.type == "transcript.text.delta":
            print(event.delta, end="", flush=True)

        elif event.type == "transcript.text.segment":
            print(
                f"\n"
                f"[{event.start:.2f}s - {event.end:.2f}s] "
                f"Speaker {event.speaker}: {event.text}"
            )

        elif event.type == "transcript.text.done":
            print("\nTranscription completed.")


[4.60s - 7.30s] Speaker A:  Thank gosh, you've already pres produced a PowerPoint presentation.

[7.30s - 9.30s] Speaker B:  Uh I think it's already on actually.

[9.30s - 9.45s] Speaker A: 

[10.55s - 10.65s] Speaker A:  Hmm.

[10.85s - 11.20s] Speaker A:  I don't know.

[12.25s - 12.60s] Speaker B:  Uh

[13.15s - 13.20s] Speaker A:  Mm.

[13.20s - 13.40s] Speaker B: 

[15.00s - 16.05s] Speaker B:  God, how do you make this thing work?

[18.15s - 18.20s] Speaker A:  Oh.

[19.35s - 19.95s] Speaker A:  Maybe the input.

[20.65s - 20.85s] Speaker B:  Uh

[20.85s - 20.90s] Speaker A:  Mm.

[20.90s - 21.10s] Speaker B: 

[33.40s - 34.85s] Speaker B:  I've plugged it in the back, but

[35.00s - 35.30s] Speaker C:  Yeah.

[38.20s - 38.90s] Speaker C:  Yep, it's got it.

[39.75s - 41.90s] Speaker B:  Okay, right. Kind of all

[43.05s - 43.30s] Speaker C:  Let's

[44.75s - 45.95s] Speaker C:  It's doing something.

[46.20s - 46.25s] Speaker C:  Yes.

[48.15s - 48.25s] Speaker C:  Yes.

[48.25